# FLAG 2027 - Notebook 10: Same-clip hypothesis (why does dev disagree with our validation?)

The evaluation plan says test pairs are made from *one video frame per second paired with the audio of the corresponding segment*. So a dev **positive** pair is probably the face frame and the voice of the **same clip** (same room, light, microphone). Our validation used positives from **different clips** of the same person - harder and different.

**This notebook**
1. Adds a second validation: positives = SAME clip (row i's face with row i's voice), negatives as before (standard / same-gender).
2. Tests a new loss: **instance matching** - each voice must pick out its OWN face frame among all faces in the batch (incl. other clips of the same person). Used only where face and voice come from the same clip (challenge + VoxCeleb2; MAV-Celeb lists pair different videos, so no).
3. Builds a submission ZIP for the best new version -> **CodaBench decides**.

Allowed by the rules: the model only looks at each test pair; no test statistics.

About 60-90 min. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive + GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev)

In [ ]:
# Cell 2 - load challenge, MAV-Celeb and VoxCeleb2 features (as Notebook 09)
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
LOCAL = '/content/flag_small'
os.makedirs(LOCAL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    flag = os.path.join(LOCAL, z + '.done')
    if not os.path.exists(flag):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(LOCAL, members=[m for m in zf.namelist() if m.endswith(('.csv', '.txt'))])
        open(flag, 'w').close()
INDEX = [os.path.join(r, f) for r, _, fs in os.walk(LOCAL) for f in fs if f.endswith(('.csv', '.txt'))]
def find_one(*parts):
    hits = [p for p in INDEX if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)

SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FEAT = {}
Xv, y = read_feat(find_one('train_English_voices.csv'), 192)
FEAT['ecapa'] = {'train': Xv}
for s in SPLIT_NAMES[1:]:
    track, lang = s.rsplit('_', 1)
    FEAT['ecapa'][s] = read_feat(find_one('/' + track + '/', lang + '_test_voices.csv'), 192)[0]
for name in ['clip', 'wavlm_sv']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in SPLIT_NAMES}
y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y)
id_gender = {c: spk2g[cls2spk[c]] for c in ids}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
VOXF = [dict(np.load(os.path.join(FEATDIR, f))) for f in sorted(os.listdir(FEATDIR)) if f.startswith('vox_') and f.endswith('.npz')]
VOX = {k: np.concatenate([v[k] for v in VOXF]) for k in ['spk', 'clip', 'ecapa', 'wavlm_sv']}
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
# sanity check of the same-clip assumption for challenge train rows: voice and face file have the same clip number
same_num = np.mean([os.path.basename(t[2]).split('.')[0] == os.path.basename(t[3]).split('.')[0] for t in train_lines])
note('Challenge train rows whose voice and face share the clip number: %.0f%%' % (100 * same_num))

In [ ]:
# Cell 3 - combined table, per-dataset normalization, model, losses, BOTH validation types
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve

def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def face_of(clip):
    return l2(clip).astype(np.float32)
def voice_of(ecapa, wavlm):
    return np.concatenate([l2(ecapa), l2(wavlm)], 1).astype(np.float32)

N_CH, N_MAV, N_VOX = len(y), len(MAV['spk']), len(VOX['spk'])
ext_spk = list(MAV['spk']) + list(VOX['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.concatenate([y, np.array([ext_lab[s] for s in ext_spk])])
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int), 2 * np.ones(N_VOX, int)]
SAMECLIP = np.r_[np.ones(N_CH, bool), np.zeros(N_MAV, bool), np.ones(N_VOX, bool)]   # face+voice from the same clip?
F_RAW = np.concatenate([face_of(FEAT['clip']['train']), face_of(MAV['clip']), face_of(VOX['clip'])])
V_RAW = np.concatenate([voice_of(FEAT['ecapa']['train'], FEAT['wavlm_sv']['train']), voice_of(MAV['ecapa'], MAV['wavlm_sv']), voice_of(VOX['ecapa'], VOX['wavlm_sv'])])
EXT = N_CH + np.arange(N_MAV + N_VOX)

def domain_normalize(tr_idx):
    F, V = F_RAW.copy(), V_RAW.copy(); ch_stats = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch_stats = st
    return F, V, ch_stats

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv)
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused), ft, vt

def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)

def instance_loss(ft, vt, mask, tau=0.1):
    # each voice must pick out ITS OWN face frame (same clip) among all faces in the batch, and vice versa.
    # uses the same space as the score (tanh embeddings, negative L2 distance as similarity)
    if mask.sum() < 2:
        return ft.sum() * 0
    f, v = ft[mask], vt[mask]
    sim = -torch.cdist(v, f) / tau
    tgt = torch.arange(len(f), device=f.device)
    return (Fn.cross_entropy(sim, tgt) + Fn.cross_entropy(sim.t(), tgt)) / 2

def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm
    ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()   # HIGHER = same person

def make_pairs(idx, n=2000, same_gender_neg=False, same_clip_pos=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); r = rng.choice(rows_[s])
        V.append(r); F.append(r if same_clip_pos else rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)

K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    vsets = [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k),                 # old: different clips
             make_pairs(va_idx, same_clip_pos=True, seed=200 + k), make_pairs(va_idx, same_gender_neg=True, same_clip_pos=True, seed=300 + k)]  # new: same clip
    FOLD_DATA.append((tr_idx, vsets))
print('ready: rows', len(Y_ALL), '| people', len(set(Y_ALL)), '| same-clip rows', int(SAMECLIP.sum()))

In [ ]:
# Cell 4 - training (bal75 recipe) + optional instance (same-clip) loss; 4 validation numbers per run
SEEDS = [1, 2, 3]
P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS, IN_FRAC = 16, 8, 2.0, 1e-4, 100, 15, 60, 0.75

def train_variant(tr_idx, lam=0.0, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch_stats = domain_normalize(all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    sc = torch.tensor(SAMECLIP[all_tr], device=dev)
    cls_in = np.zeros(len(cl), bool); cls_in[np.unique(ytr_np[D_ALL[all_tr] < 2])] = True
    in_cls, out_cls = np.where(cls_in)[0], np.where(~cls_in)[0]; n_in = int(round(P_ID * IN_FRAC))
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev)
    model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6)
    mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01)
    best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = np.r_[rng.choice(in_cls, n_in, replace=False), rng.choice(out_cls, P_ID - n_in, replace=False)]
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits, ft, vt = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j])
            if lam > 0:
                loss = loss + lam * instance_loss(ft, vt, sc[j])
            opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])   # mean of all 4
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch_stats

VARIANTS = {'S0_bal75': 0.0, 'S1_instance_0.3': 0.3, 'S2_instance_1.0': 1.0}
CSV = os.path.join(RESDIR, 'nb10_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['variant', 'seed', 'fold'])
t0 = time.time()
for name, lam in VARIANTS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.variant == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]
            model, best_ep, XF, XV, _ = train_variant(tr_idx, lam, vsets, seed=seed)
            e = [eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets]
            row = pd.DataFrame([dict(variant=name, seed=seed, fold=k, best_ep=best_ep,
                                     std_diffclip=e[0], gen_diffclip=e[1], std_sameclip=e[2], gen_sameclip=e[3])])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
            done = pd.concat([done, row])
            print('%-16s seed %d fold %d | ep %3d | diff-clip std %.2f gen %.2f | SAME-clip std %.2f gen %.2f | %.0f min'
                  % (name, seed, k, best_ep, e[0], e[1], e[2], e[3], (time.time() - t0) / 60))

In [ ]:
# Cell 5 - results table
R = pd.read_csv(CSV)
R['overall_diffclip'] = (R.std_diffclip + R.gen_diffclip) / 2
R['overall_sameclip'] = (R.std_sameclip + R.gen_sameclip) / 2
cols = ['std_diffclip', 'gen_diffclip', 'overall_diffclip', 'std_sameclip', 'gen_sameclip', 'overall_sameclip']
per_seed = R.groupby(['variant', 'seed'])[cols].mean().reset_index()
tab = per_seed.groupby('variant')[cols].mean()
tab = tab.loc[[v for v in VARIANTS if v in tab.index]].round(2)
note('SAME-CLIP HYPOTHESIS (validation on held-out CHALLENGE people, 5-fold x 3 seeds)')
note('  diffclip = positives from DIFFERENT clips (our old validation) | sameclip = positives from the SAME clip (like dev, probably)')
note(tab.to_string())
note('Reference CodaBench dev for S0 (bal75) = 31.24')

In [ ]:
# Cell 6 - final model for the best instance-loss version (by same-clip validation) -> submission ZIP
LOWER_IS_SAME = True
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
cand = tab.loc[[v for v in VARIANTS if v != 'S0_bal75' and v in tab.index]]
BEST = cand['overall_sameclip'].idxmin()
lam = VARIANTS[BEST]
ep = int(np.median(R[R.variant == BEST]['best_ep']))
finals = []
for s in SEEDS:
    m, _, _, _, ch_stats = train_variant(np.arange(N_CH), lam, None, seed=s, max_epochs=ep)
    finals.append(m)
OUT = '/content/sub_10_' + BEST
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang, fname in LANG_FILE.items():
        split = track + '_' + lang
        pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang + '_test.txt')) if l.strip()]
        fx = ((face_of(FEAT['clip'][split]) - ch_stats[0]) / ch_stats[1]).astype(np.float32)        # challenge TRAIN stats
        vx = ((voice_of(FEAT['ecapa'][split], FEAT['wavlm_sv'][split]) - ch_stats[2]) / ch_stats[3]).astype(np.float32)
        xf = torch.tensor(fx, device=dev); xv = torch.tensor(vx, device=dev)
        s_ = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
        out = -s_ if LOWER_IS_SAME else s_
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, out):
                f.write('%s %.6f\n' % (pid, v))
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
ZIP_PATH = os.path.join(SUBDIR, '10_' + BEST.replace('.', 'p') + '.zip')
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
note('Submission ZIP saved: %s (%d epochs)' % (ZIP_PATH[len(BASE):], ep))

LOG = os.path.join(PROJECT, 'experiments.csv')
log = pd.read_csv(LOG) if os.path.exists(LOG) else pd.DataFrame()
if len(log) and 'run' in log:
    log['codabench_dev'] = log['codabench_dev'].astype(object)
    log.loc[log.run == '08_E3_vox1236_dn_bal75', 'codabench_dev'] = 31.24
    log = log[~log.run.isin(['10_' + v for v in VARIANTS])]
new = pd.DataFrame([{'date': time.strftime('%Y-%m-%d %H:%M'), 'run': '10_' + v, 'change': 'bal75 + instance loss lambda=%s' % VARIANTS[v],
                     'seed': '1,2,3', 'cv_std': tab.loc[v, 'std_diffclip'], 'cv_gender': tab.loc[v, 'gen_diffclip'],
                     'cv_overall': tab.loc[v, 'overall_diffclip'], 'codabench_dev': ''} for v in VARIANTS if v in tab.index])
pd.concat([log, new], ignore_index=True).to_csv(LOG, index=False)
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)